# Обзор датасетов: «Балтийский баланс — 2030»

**Рабочий ноутбук Data Engineer.** Запускайте ячейки сверху вниз: `Run → Run All Cells`.
За один проход вы получите состав данных, источники, периоды, пропуски, графики и таблицу пригодности для команды.

Здесь исследуются **опубликованные показатели и ограничения их покрытия**. Выбор цели прогноза остаётся за командой.
Годовые значения сохраняются годовыми, отсутствующие значения не заменяются нулями.
Внешняя торговля региона, портовая перевалка, транзитные записи и мощность терминала рассматриваются отдельно.

**Перед первым запуском:** в терминале из корня проекта выполните `pip install -e ".[notebooks]"`,
выберите Python из `.venv`, затем `python -m pipeline.build` и `python -m pipeline.verify`.
Нужны оригиналы и подготовленные CSV из рабочей папки или архива передачи; одного клона Git недостаточно.
Интернет требуется для установки пакетов, сам обзор читает только локальные файлы.

Маршрут: **проверка комплекта → инвентаризация → источники → качество → паромная линия → контекст → экспорт → транзит → выводы**.


## 1. Окружение и проверка комплекта

Ячейки не скачивают файлы и не изменяют исходные таблицы. Корень проекта находится автоматически,
если Jupyter открыт из корня или папки `notebooks/`. При изменении исходников сначала пересоберите данные в терминале.


In [ ]:
import csv
import json
import sys
from decimal import Decimal
from pathlib import Path

try:
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import Markdown, display
except ImportError as error:
    raise ImportError('В выбранном Python нужны зависимости: pip install -e ".[notebooks]"') from error

get_ipython().run_line_magic("matplotlib", "inline")
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 30)
pd.set_option("display.max_colwidth", 85)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.2,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.family": "DejaVu Sans", "figure.dpi": 110})

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "data/source_registry.json").is_file() and (p / "pipeline").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Откройте Jupyter из корня проекта или папки notebooks; нужен data/source_registry.json.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Проект:", ROOT)
print("Python:", sys.executable)
print("pandas:", pd.__version__)


In [ ]:
from pipeline.verify import verify

manifest_path = ROOT / "data/processed/build_manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError("Нет подготовленных таблиц. Получите оригиналы и выполните python -m pipeline.build.")
verification_errors = verify(ROOT)
if verification_errors:
    raise ValueError("Комплект требует проверки. После изменения кода выполните python -m pipeline.build.\n" +
                     "\n".join(verification_errors[:10]))
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
registry = json.loads((ROOT / "data/source_registry.json").read_text(encoding="utf-8"))
print("Версия данных:", manifest["data_version"])
print("Целевой ряд:", manifest["target_selection"], "— ожидает решения команды")
print("Хеши, происхождение, единицы и согласованность сводок: проверены.")


## 2. Какие таблицы есть

Число строк не равно числу временных точек или перевозок. Например, два направления по девять лет —
это две отдельные истории по девять точек; у одной транзитной декларации бывает несколько товаров.
`exports_kaliningrad.csv` — подвыборка `exports_clean.csv`, а месячная сводка транзита построена из товарных строк.
Эти производные файлы не добавляют независимых наблюдений.

**Где лежат таблицы и откуда они получены.** Все пути ниже указаны **от корня проекта**.
Сохранённые оригиналы перечислены для комплекта `bb-data-015224e90bac`; связь конкретной строки с оригиналом — через `source_id` в `data/source_registry.json`.

| Файл в проекте | Происхождение |
|---|---|
| `data/processed/candidate_observations.csv` | [Росморпорт: погрузочно-разгрузочная деятельность](https://www.rosmorport.ru/filials/spb_serv_freight/); строки паромной линии, раздел 5 |
| `data/processed/context_observations.csv` | [Деятельность терминала](https://www.rosmorport.ru/filials/spb_serv_freight/) и [перевозки филиала](https://www.rosmorport.ru/filials/spb_serv_ferry/); раздел 6 |
| `data/processed/coverage.csv` | Рассчитано из `candidate_observations.csv` и `context_observations.csv`; отдельного внешнего источника нет |
| `data/processed/scenario_parameters.csv` | Заявленные мощности объектов на [странице Росморпорта](https://www.rosmorport.ru/filials/spb_serv_freight/); раздел 6 |
| `data/processed/factors.csv` | Пока только схема: исходные значения факторов ещё не добавлены |
| `data/processed/exports_clean.csv` | Три присланных XLSX: ЭК, НЭ, ННЭ; пути к оригиналам в разделе 7 |
| `data/processed/exports_kaliningrad.csv` | Фильтр `exports_clean.csv` по Калининградской области |
| `data/processed/export_growth_checks.csv` | Сверка значений и опубликованных индексов из тех же трёх XLSX; раздел 7 |
| `data/processed/transit_items_clean.csv` | Присланный `2204_tranzit_EN.xls`; путь к оригиналу в разделе 8 |
| `data/processed/transit_monthly_summary.csv` | Группировка `transit_items_clean.csv` по месяцу оформления, странам, транспорту и товарной группе |
| `data/processed/table_row_issues.csv` | Журнал обработки экспортных и транзитного файлов; отдельного внешнего источника нет |
| `data/interim/reconciliation.csv` | Диагностическая разность показателей линии и терминала из сохранённой страницы Росморпорта |
| `data/source_catalog.csv` | [Предоставленная Google-таблица](https://docs.google.com/spreadsheets/d/1lWth7AgxGWDGiAHEXNgSjNQTaCmZDxHrBlmADg6GWLQ/edit); оригинал XLSX указан в разделе 3 |



In [ ]:
# Имена кодов, стран и товарных групп читаются строками: ведущие нули сохраняются.
# keep_default_na=False не превращает код страны NA или текст ND в пропуск.
relative_csvs = sorted(p for p in manifest["outputs"] if p.endswith(".csv"))
tables = {Path(p).name: pd.read_csv(ROOT / p, dtype="string", encoding="utf-8-sig", keep_default_na=False)
          for p in relative_csvs}

dataset_roles = {
    "candidate_observations.csv": "Кандидат цели: узкая паромная линия, два направления",
    "context_observations.csv": "Контекст: терминал и перевозки филиала, иной охват",
    "coverage.csv": "Аудит покрытия отдельных рядов",
    "exports_clean.csv": "Внешняя торговля всех представленных территорий",
    "exports_kaliningrad.csv": "Подвыборка области из exports_clean",
    "export_growth_checks.csv": "Диагностика индексов роста, не новые наблюдения",
    "transit_items_clean.csv": "Товарные строки выборки транзитных деклараций",
    "transit_monthly_summary.csv": "Производная сводка товарных строк, не уникальные отправки",
    "scenario_parameters.csv": "Заявленные мощности объектов, не факт перевозки",
    "factors.csv": "Заготовка факторов; доступность и лаги пока не подтверждены",
    "table_row_issues.csv": "Журнал повторов и конфликтов ключей",
    "reconciliation.csv": "Диагностика различия охвата линии и терминала",
    "source_catalog.csv": "Каталог ссылок; это не таблица проверенных фактов",
}

def period_range(frame):
    for column in ("period", "registered_date"):
        if column in frame:
            values = frame.loc[frame[column].ne(""), column]
            return f"{values.min()} — {values.max()}" if not values.empty else "нет значений"
    return "не временной ряд"

inventory = pd.DataFrame([
    {"file": name, "rows": len(frame), "columns": len(frame.columns),
     "period_range": period_range(frame), "role": dataset_roles.get(name, "Дополнительная таблица")}
    for name, frame in tables.items()
])
display(inventory)


### Посмотреть любую таблицу

Поменяйте `PREVIEW_DATASET` и число строк. Справочник всех столбцов ниже помогает понять схему;
полные названия, единицы и правила агрегации описаны в `docs/contracts.md`.

После выбора файла ячейка выводит его **полный путь на вашей машине**, а при наличии `source_id` —
адрес получения и полный путь к каждому использованному оригиналу. Для производных таблиц без `source_id`
происхождение указано в таблице раздела 2.



In [ ]:
PREVIEW_DATASET = "exports_kaliningrad.csv"
PREVIEW_ROWS = 6
preview = tables[PREVIEW_DATASET]
display(pd.DataFrame({"column": preview.columns, "storage_type": preview.dtypes.astype(str).values}))
display(preview.head(PREVIEW_ROWS))

# Путь выбранного CSV и оригиналы именно тех source_id, которые есть в этой таблице.
preview_relative = next(p for p in relative_csvs if Path(p).name == PREVIEW_DATASET)
print("Обработанная таблица:", ROOT / preview_relative)
if "source_id" in preview.columns:
    preview_source_ids = set(preview.loc[preview["source_id"].ne(""), "source_id"])
    for artifact in registry["artifacts"]:
        if artifact["source_id"] in preview_source_ids:
            origin = artifact["url"]
            label = "Передан пользователем: " + origin.removeprefix("provided:") if origin.startswith("provided:") else origin
            print("\nИсточник:", label)
            print("Оригинал:", ROOT / artifact["local_path"])
            print("source_id:", artifact["source_id"])
else:
    print("Происхождение этой производной таблицы указано в разделе 2.")


## 3. Откуда данные и что ещё не изучено

Реестр содержит сохранённые оригиналы, включая страницы, PDF, XLS/XLSX и инструмент чтения XLS.
Наличие PDF или страницы в реестре не означает, что из них уже получен датасет.
Поля `purpose` описывают назначение источника; `observed`, `estimated`, `synthetic` описывают статус значения — это разные признаки.

**Откуда взят каталог:** [исходная Google-таблица](https://docs.google.com/spreadsheets/d/1lWth7AgxGWDGiAHEXNgSjNQTaCmZDxHrBlmADg6GWLQ/edit),
[адрес скачивания XLSX](https://docs.google.com/spreadsheets/d/1lWth7AgxGWDGiAHEXNgSjNQTaCmZDxHrBlmADg6GWLQ/export?format=xlsx). Сохранённый оригинал: `data/raw/hackathon_catalog/11a6137c973b4089f83d02d022aec7f286dabcc9c3f48f9d01197f9dd7e35198.xlsx`.
Из него получен `data/source_catalog.csv`.

**Где проверять остальные источники:** `data/source_registry.json`.
В нём `url` — адрес или указание на переданный файл, `local_path` — путь к сохранённому оригиналу,
`source_id` — идентификатор, указанный в строках обработанных таблиц.



In [ ]:
sources = pd.DataFrame(registry["artifacts"])
source_columns = ["source_key", "organization", "purpose", "period", "original_unit", "url"]
display(sources[source_columns])
print("Сохранённых артефактов:", len(sources))
print("Уникальных source_id:", sources["source_id"].nunique())
# Для проверки конкретного оригинала: sources[["source_id", "local_path", "sha256"]]


In [ ]:
# Полный каталог и неудачные получения сохраняются отдельно от проверенных таблиц.
catalog = tables.get("source_catalog.csv", pd.DataFrame())
if not catalog.empty:
    display(catalog.groupby(["purpose", "review_status"], dropna=False).size()
            .rename("links").reset_index())
    display(catalog[["title", "purpose", "review_status", "url"]])
else:
    print("Каталог ссылок отсутствует в этом комплекте.")
failures = pd.DataFrame(registry.get("failures", []))
if failures.empty:
    print("Ошибок получения в реестре нет.")
else:
    display(failures)
print("Ошибка доступа означает непроверенный источник; проверка TLS не отключается.")


## 4. Пропуски, повторы и единицы

Сначала считаем пустые ячейки. Пустой `direction` у контекста — отсутствие разбивки, а не направление с нулевым потоком.
Пустые даты действия мощности означают неизвестный срок. Пустой `factors.csv` — ещё не подготовленные факторы.

Ниже показаны **повторения ключей**, а не доказанные дубли физических отправок.
Хеши оригиналов и результаты нормализации уже проверены первой ячейкой через `pipeline.verify`.

**Файлы проверок:** `data/processed/table_quality.json` — профиль качества пользовательских файлов;
`data/processed/table_row_issues.csv` — журнал повторов и конфликтов;
`data/processed/build_manifest.json` — версия, хеши и состав комплекта.
Таблица пропусков ниже рассчитывается из уже загруженных CSV; это результат проверки, а не новый внешний датасет.



In [ ]:
missing_rows = []
for name, frame in tables.items():
    for column in frame.columns:
        missing = int(frame[column].eq("").sum())
        if missing:
            missing_rows.append({"file": name, "column": column, "missing": missing,
                                 "rows": len(frame), "missing_pct": round(missing / len(frame) * 100, 2)})
missing_profile = pd.DataFrame(missing_rows, columns=["file", "column", "missing", "rows", "missing_pct"])
display(missing_profile.sort_values(["missing_pct", "file"], ascending=[False, True]))
empty_names = [name for name, frame in tables.items() if frame.empty]
print("Таблицы только с заголовком:", ", ".join(empty_names) or "нет")


In [ ]:
key_columns = {
    "candidate_observations.csv": ["period", "direction", "mode", "cargo_group", "source_id", "route_id"],
    "context_observations.csv": ["series_id", "period", "source_id"],
    "exports_clean.csv": ["export_scope", "reporting_geography", "enterprise_group", "period", "source_id"],
    "exports_kaliningrad.csv": ["export_scope", "reporting_geography", "enterprise_group", "period", "source_id"],
    "transit_items_clean.csv": ["declaration_id_hash", "item_number"],
    "transit_monthly_summary.csv": ["period", "origin_country_code", "destination_country_code", "actual_mode", "commodity_prefix", "source_id"],
}
key_checks = pd.DataFrame([
    {"file": name, "rows_in_repeated_keys": int(tables[name].duplicated(keys, keep=False).sum()),
     "key": " + ".join(keys)} for name, keys in key_columns.items() if name in tables
])
display(key_checks)
issues = tables.get("table_row_issues.csv", pd.DataFrame())
if issues.empty:
    print("Журнал точных повторов и конфликтов пуст.")
else:
    display(issues)

def numeric_column(frame, column):
    # Ошибочный текст не превращается незаметно в NaN.
    return pd.to_numeric(frame[column].replace("", pd.NA), errors="raise")

figures = {}


## 5. Кандидат цели: паромная линия Усть-Луга — Калининград


**Источник данных:** [ФГУП «Росморпорт», «Погрузочно-разгрузочная деятельность»](https://www.rosmorport.ru/filials/spb_serv_freight/).
Использованы строки «Выгружено (тонн)» и «Погружено (тонн)» раздела паромной линии за 2015–2023 годы.

- Сохранённая страница: `data/raw/rosmorport_freight/44a0339134dfbf750816304f99d26ed9167b5bc96d0effb76aa53629f820b524.html`.
- Подготовленная таблица: `data/processed/candidate_observations.csv`.
- Покрытие ряда: `data/processed/coverage.csv`, рассчитано из подготовленных наблюдений.
- Паспорт: `source_id = rosmorport_freight__44a0339134df`; `source_locator` в CSV указывает исходную строку/ячейку.



In [ ]:
candidate = tables["candidate_observations.csv"].copy()
candidate["year"] = pd.to_numeric(candidate["period"], errors="raise").astype(int)
candidate["value_tonnes"] = numeric_column(candidate, "value_tonnes")
assert candidate["status"].eq("observed").all(), "В кандидате есть значения с другим статусом"
direction_names = {"inbound": "В область", "outbound": "Из области"}
coverage = candidate.groupby("direction").agg(
    observations=("value_tonnes", "count"), time_points=("year", "nunique"),
    first_year=("year", "min"), last_year=("year", "max"))
display(coverage)
display(tables["coverage.csv"])

fig, ax = plt.subplots()
for direction, group in candidate.groupby("direction"):
    group = group.sort_values("year")
    ax.plot(group["year"], group["value_tonnes"] / 1000, marker="o", label=direction_names.get(direction, direction))
ax.set(title="Опубликованные показатели узкой линии; кандидат цели", xlabel="Год", ylabel="Масса, тыс. т за год")
ax.set_xticks(sorted(candidate["year"].unique()))
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
fig.tight_layout()
figures["ferry_candidate"] = fig
plt.show()


**Что описать команде:** длина истории по каждому направлению, отсутствие месяцев и групп грузов,
изменение после 2022 года и вопросы определения массы. График не устанавливает причины изменений и точность прогноза.


## 6. Справочные ряды, мощности и факторы

Портовый итог, каботаж и перевозки филиала имеют разные объекты учёта.
Показываем каждый ряд отдельно, без общей суммы и без продолжения направленного кандидата числами другого охвата.

**Источники и файлы:**

- Показатели терминала (`terminal_total`, `terminal_cabotage`) — [страница погрузочно-разгрузочной деятельности Росморпорта](https://www.rosmorport.ru/filials/spb_serv_freight/).
  Оригинал: `data/raw/rosmorport_freight/44a0339134dfbf750816304f99d26ed9167b5bc96d0effb76aa53629f820b524.html`.
- Перевозки филиала (`ferry_branch_main`, `ferry_branch_spb_call`) — [страница «Паромные перевозки» Росморпорта](https://www.rosmorport.ru/filials/spb_serv_ferry/).
  Оригинал: `data/raw/rosmorport_ferry/bd1008b9ce3f27df77509b70c9006020f52f5ebd69fefe16f691a1b597124d87.html`.
- Оба источника извлечены в `data/processed/context_observations.csv`; конкретный оригинал каждой строки определяется по `source_id`.
- Мощности ЖДПК и терминала Пионерский взяты из описания объектов на первой странице и записаны в `data/processed/scenario_parameters.csv`.
- `data/processed/factors.csv` пока содержит только заголовок, поэтому первоисточник значений факторов ещё не назначен.
- `data/interim/reconciliation.csv` рассчитан из показателей линии и каботажа терминала; самостоятельного внешнего источника у этой разности нет.



In [ ]:
context = tables["context_observations.csv"].copy()
context["value_tonnes"] = numeric_column(context, "value_tonnes")
context["year"] = pd.to_numeric(context["period"], errors="raise").astype(int)
display(context[["series_id", "period", "metric", "original_value", "original_unit", "value_tonnes", "quality_note"]])
series_labels = {"ferry_branch_main": "Филиал: основная таблица", "ferry_branch_spb_call": "Филиал: дополнительная таблица",
                 "terminal_cabotage": "Терминал: каботаж", "terminal_total": "Терминал: весь объём"}
series_ids = sorted(context["series_id"].unique())
fig, axes = plt.subplots(len(series_ids), 1, figsize=(10, 2.7 * len(series_ids)), squeeze=False)
for ax, series_id in zip(axes.flat, series_ids):
    group = context.loc[context["series_id"].eq(series_id)].sort_values("year")
    # NaN сохраняет разрыв линии, а не нулевой объём.
    ax.plot(group["year"], group["value_tonnes"].to_numpy(dtype=float, na_value=float("nan")) / 1000, marker="o")
    ax.set(title=series_labels.get(series_id, series_id), xlabel="Год", ylabel="Масса, тыс. т за год")
    ax.set_xticks(group["year"])
fig.suptitle("Справочные показатели: каждый ряд отдельно", y=1.01)
fig.tight_layout()
figures["context_series"] = fig
plt.show()


In [ ]:
display(tables["scenario_parameters.csv"])
print("Мощности в tonnes/year и trucks/year не складываются. Даты действия и доступность не установлены.")
factors = tables["factors.csv"]
print("Подготовленных строк факторов:", len(factors))
if not factors.empty:
    display(factors)
reconciliation = tables.get("reconciliation.csv")
if reconciliation is not None:
    display(reconciliation)
print("Разность с терминалом — проверка охвата показателей, а не поправка массы линии.")


## 7. Три экспортных XLSX

**ЭК** — весь экспорт; **НЭ** — несырьевой; **ННЭ** — несырьевой неэнергетический.
Это вложенные виды, их нельзя складывать. Итог страны, округа и регионы также перекрываются.
«Всего» не складывается с микро-, малыми и средними предприятиями.

В колонках только **январь 2021 и январь 2022**. Заголовок 2020–2022 у НЭ/ННЭ не добавляет данных за 2020 год.
Это внешняя торговля; нет разбивки по направлениям и видам транспорта нужного коридора.


| Переданный файл | Путь к сохранённому оригиналу | Вид экспорта |
|---|---|---|
| `ЭК_январь_2022.xlsx` | `data/raw/exports_all_jan2022/13466a6332f767c334cd96e7ffaf3670a4a57f06c8b69ff03f4d3bf46a803c0e.xlsx` | Весь экспорт |
| `НЭ_январь_2022.xlsx` | `data/raw/exports_non_raw_jan2022/19d8c318219e179fcb0e2fd35c1e7ab03ad17343d75dbc8d5718e0f488669ac2.xlsx` | Несырьевой экспорт |
| `ННЭ_январь_2022.xlsx` | `data/raw/exports_non_raw_non_energy_jan2022/f831fda79ff99d600ac86e394f86513d2ff5658b7c1771cd01b9114bb3d9549c.xlsx` | Несырьевой неэнергетический экспорт |

**После обработки:** `data/processed/exports_clean.csv` — все территории;
`data/processed/exports_kaliningrad.csv` — область.
В CSV `source_id` связывает строку с одним из трёх оригиналов, а `source_locator` показывает лист и диапазон ячеек.



In [ ]:
exports = tables["exports_clean.csv"].copy()
for column in ("enterprise_count", "weight_tonnes", "trade_value_thousand_usd", "trade_value_usd"):
    exports[column] = numeric_column(exports, column)
export_coverage = exports.groupby(["export_scope", "geography_level"]).agg(
    rows=("period", "size"), geographies=("reporting_geography", "nunique"), periods=("period", lambda s: ", ".join(sorted(s.unique()))))
display(export_coverage)
kaliningrad = exports.loc[exports["reporting_geography"].eq("КАЛИНИНГРАДСКАЯ ОБЛАСТЬ")].copy()
display(kaliningrad[["export_scope", "enterprise_group", "period", "enterprise_count", "weight_tonnes", "trade_value_thousand_usd"]])


### Сравнение двух январей по Калининградской области

Берём только группу предприятий **«Всего»**. Каждый вид экспорта показываем отдельной категорией,
массу и стоимость — на разных графиках. Это сравнение двух наблюдений, а не прогноз или длинный тренд.

**Данные этого сравнения:** `data/processed/exports_kaliningrad.csv`.
В коде та же выборка получается фильтром `data/processed/exports_clean.csv`, затем оставляется группа «Всего».
Оригиналы — три XLSX, перечисленные выше; другого источника у графиков нет.



In [ ]:
region_totals = kaliningrad.loc[kaliningrad["enterprise_group"].eq("Всего")].copy()
scope_labels = {"all_exports": "Весь экспорт", "non_raw_exports": "Несырьевой", "non_raw_non_energy_exports": "Несырьевой неэнергетический"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for metric, ax, label in [("weight_tonnes", axes[0], "Масса, т за январь"),
                           ("trade_value_thousand_usd", axes[1], "Стоимость, тыс. USD за январь")]:
    pivot = region_totals.pivot(index="export_scope", columns="period", values=metric).reindex(scope_labels)
    pivot.index = pivot.index.map(scope_labels).str.replace("Несырьевой неэнергетический", "Несырьевой\nнеэнергетический", regex=False)
    pivot.plot.bar(ax=ax, rot=0)
    ax.set(xlabel="", ylabel=label)
    ax.tick_params(axis="x", labelsize=8)
    ax.legend(title="Месяц")
    ax.set_ylim(0, float(pivot.max().max()) * 1.35)
fig.suptitle("Калининградская область: вложенные виды экспорта; категории не складывать")
fig.tight_layout()
figures["kaliningrad_exports"] = fig
plt.show()

# Изменение рассчитывается отдельно по каждой метрике и виду экспорта.
comparison_rows = []
for metric in ("weight_tonnes", "trade_value_thousand_usd", "enterprise_count"):
    values = region_totals.pivot(index="export_scope", columns="period", values=metric)
    periods = sorted(values.columns)
    if len(periods) == 2:
        base, current = periods
        change = (values[current] / values[base].where(values[base].ne(0)) - 1) * 100
        for scope in values.index:
            comparison_rows.append({"export_scope": scope, "metric": metric,
                                    "base_period": base, "current_period": current,
                                    "base_value": values.at[scope, base], "current_value": values.at[scope, current],
                                    "change_pct": change.loc[scope]})
export_comparison = pd.DataFrame(comparison_rows)
display(export_comparison.round(2))


### Проверка опубликованных индексов

«Темп роста 71,4%» означает 0,714 от базы, то есть снижение примерно на 28,6%.
Индекс и процент изменения — разные величины.
Расхождения сравниваются с видимыми округлёнными числами; они не доказывают ошибку первоисточника.
Порог 0,15 процентного пункта — диагностическое правило, не статистический критерий качества.

**Файл диагностики:** `data/processed/export_growth_checks.csv`.
Базовые значения, текущие значения и опубликованные индексы взяты из тех же трёх XLSX раздела 7;
отношение и разность вычислены обработчиком. `source_id` и `source_locator` позволяют найти исходную ячейку.



In [ ]:
growth = tables["export_growth_checks.csv"]
growth_counts = growth.groupby(["export_scope", "check_status"]).size().unstack(fill_value=0)
display(growth_counts)
# fill_value=0 здесь означает отсутствие записей в категории диагностики; факты не заполняются нулями.
display(growth.loc[growth["check_status"].eq("displayed_values_differ"),
                   ["export_scope", "reporting_geography", "enterprise_group", "metric", "base_value", "current_value",
                    "reported_growth_index_pct", "calculated_growth_index_pct", "difference_pp", "source_locator"]].head(12))
print("При нулевой базе расчётный индекс оставлен пустым.")


## 8. Транзитный XLS: что именно можно исследовать

Строка — **товарная позиция декларации**, не уникальная физическая отправка.
Файл содержит товары с кодом 2204 (вино), оформление конца 2016 года и завершение до января 2017.
Страны и таможни не доказывают соответствие Калининградскому коридору.

В очищенном CSV нет реквизитов участников и транспорта; хеш связывает товары одной декларации.
Смотрим количество записей, пропуски, даты и состав выборки.
`rail` не означает подтверждённое сухопутное пересечение коридора, `water` сохраняет неоднозначность «морской/речной».

**Откуда взяты данные:** файл `2204_tranzit_EN.xls`, присланный вами в чат.
Организация-составитель, сайт публикации и методика отбора пока не установлены.

- Неизменённый оригинал: `data/raw/customs_transit_2204/b1ca7b748adde5e3ec68a978cb9b1237ab5a45c8bbd75e509daff181bc0d3fa6.xls`; лист `Транзит`.
- Очищенные товарные строки: `data/processed/transit_items_clean.csv`.
- Производная месячная сводка: `data/processed/transit_monthly_summary.csv`.
- Паспорт: `source_id = customs_transit_2204__b1ca7b748add`; `source_locator` указывает строку оригинала.

Ноутбук читает очищенный CSV. Исходные реквизиты участников и транспорта остаются в оригинале и не выводятся в обзор.



In [ ]:
transit = tables["transit_items_clean.csv"].copy()
for column in ("item_gross_kg", "item_gross_tonnes", "item_net_kg", "item_net_tonnes", "item_invoice_usd", "transit_days"):
    transit[column] = numeric_column(transit, column)
for column in ("registered_date", "completed_date"):
    transit[column] = pd.to_datetime(transit[column].replace("", pd.NA), format="%Y-%m-%d", errors="raise")
transit_profile = pd.DataFrame([
    {"metric": "Товарных строк", "value": len(transit)},
    {"metric": "Уникальных деклараций в выборке", "value": transit["declaration_id_hash"].nunique()},
    {"metric": "Начало регистрации", "value": str(transit["registered_date"].min().date())},
    {"metric": "Конец регистрации", "value": str(transit["registered_date"].max().date())},
    {"metric": "Последнее завершение", "value": str(transit["completed_date"].max().date())},
    {"metric": "Нетто отсутствует, строк", "value": int(transit["item_net_tonnes"].isna().sum())},
    {"metric": "Нетто отсутствует, %", "value": round(transit["item_net_tonnes"].isna().mean() * 100, 2)},
])
display(transit_profile)
display(transit["commodity_code"].str[:4].value_counts().rename_axis("commodity_prefix").rename("item_rows").reset_index())
display(transit[["registered_date", "completed_date", "origin_country_code", "destination_country_code", "actual_mode",
                 "commodity_code", "item_gross_tonnes", "item_net_tonnes", "quality_flags"]].head(8))


In [ ]:
mode_names = {"road": "Автодорога", "rail": "Железная дорога", "water": "Морской/речной", "air": "Авиа", "other": "Другой", "": "Не указан"}
mode_counts = transit["actual_mode"].value_counts().rename_axis("mode").rename("item_rows").reset_index()
mode_counts["share_pct"] = mode_counts["item_rows"] / len(transit) * 100
display(mode_counts.round(2))
country_pairs = transit.groupby(["origin_country_code", "destination_country_code"]).size().sort_values(ascending=False).rename("item_rows").reset_index()
display(country_pairs.head(10))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
labels = mode_counts["mode"].map(mode_names)
mode_bars = axes[0].barh(labels, mode_counts["item_rows"])
axes[0].bar_label(mode_bars, padding=3, fmt="%d")
axes[0].set_xlim(0, float(mode_counts["item_rows"].max()) * 1.12)
axes[0].set(title="Состав выборки по транспорту", xlabel="Товарные строки, шт.")
net_presence = transit["item_net_tonnes"].notna().value_counts().reindex([True, False], fill_value=0)
axes[1].bar(["Нетто известно", "Нетто отсутствует"], net_presence.values, color=["#287a64", "#bf5b37"])
axes[1].set(title="Покрытие массы нетто", ylabel="Товарные строки, шт.")
fig.tight_layout()
figures["transit_composition"] = fig
plt.show()
print("Это доли записей выборки, а не доли транспортного рынка или тоннажа коридора.")


### Срок между оформлением и завершением

Чтобы декларация с несколькими товарами не получала больший вес, сначала проверяем одинаковость дат
внутри декларации и оставляем одну запись на декларацию. Это календарный интервал между двумя полями;
он не равен чистому времени движения, простою или эффекту транспортного решения.

**Исходные поля:** `registered_date` и `completed_date` из `data/processed/transit_items_clean.csv`.
Они получены из полей «Дата оформления декларации» и «Дата завершения транзита» листа `Транзит`
в `2204_tranzit_EN.xls`. Календарный интервал рассчитывается в этой ячейке; отдельного источника длительности нет.



In [ ]:
date_columns = ["registered_date", "completed_date"]
date_consistency = transit.groupby("declaration_id_hash")[date_columns].nunique(dropna=False)
if date_consistency.gt(1).any(axis=None):
    raise ValueError("Внутри одной декларации разные даты; распределение требует отдельного разбора.")
declaration_dates = transit.drop_duplicates("declaration_id_hash").copy()
declaration_dates["calendar_days"] = (declaration_dates["completed_date"] - declaration_dates["registered_date"]).dt.days
known_days = declaration_dates["calendar_days"].dropna()
print("Деклараций с двумя датами:", len(known_days), "из", len(declaration_dates))
display(known_days.describe(percentiles=[0.5, 0.9, 0.95]).to_frame("calendar_days"))
fig, ax = plt.subplots()
ax.hist(known_days.astype(float), bins=20, edgecolor="white")
ax.set(title="Интервал между оформлением и завершением: одна запись на декларацию",
       xlabel="Календарные дни", ylabel="Декларации, шт.")
fig.tight_layout()
figures["transit_calendar_days"] = fig
plt.show()


### Сводка товарных масс

Суммируются веса **товарных позиций**, а не повторяющийся общий вес декларации.
Даже разные декларации могут отражать этапы движения одного груза, поэтому результат не является уникальным тоннажем.
Отсутствующие веса нетто не заменены брутто. В месячной таблице стоит **месяц оформления**, а не завершения.
Количество деклараций между группами складывать нельзя.

**Подробные записи:** `data/processed/transit_items_clean.csv`.
**Сводка для сверки:** `data/processed/transit_monthly_summary.csv`.
Оба файла происходят из одного `2204_tranzit_EN.xls`; сводка не добавляет независимых наблюдений.



In [ ]:
monthly_summary = tables["transit_monthly_summary.csv"]
display(monthly_summary.head(12))
# Дополнительная точная сверка масштаба без округления float.
raw_items = tables["transit_items_clean.csv"]
def decimal_sum(values):
    known = [Decimal(v) for v in values if v != ""]
    return sum(known, Decimal(0)) if known else None

mass_checks = []
for kind in ("gross", "net"):
    item_sum = decimal_sum(raw_items[f"item_{kind}_tonnes"])
    grouped_sum = decimal_sum(monthly_summary[f"item_{kind}_sum_tonnes"])
    assert item_sum == grouped_sum, f"Сводка не совпадает по {kind}"
    mass_checks.append({"metric": kind, "known_item_rows": int(raw_items[f"item_{kind}_tonnes"].ne("").sum()),
                        "sum_item_tonnes": str(item_sum) if item_sum is not None else "нет значений",
                        "summary_matches": item_sum == grouped_sum})
mass_checks = pd.DataFrame(mass_checks)
display(mass_checks)


## 9. Что передать команде

Таблица ниже — оценка пригодности имеющихся материалов, не утверждение цели и не выбор гипотез.
Для факторов нужны дата публикации/доступности и лаг, для сценарных параметров — объект и период действия.

**На чём основана оценка:** таблицы разделов 5–8 и их паспорта в `data/source_registry.json`.
Подробные пояснения: `docs/data-audit.md` и `docs/tables-quality.md`.
Рекомендации в этой таблице составлены при анализе данных с помощью ИИ; это не значения из внешней публикации.



In [ ]:
suitability = pd.DataFrame([
    {"dataset": "candidate_observations.csv", "use": "Кандидат узкой годовой цели", "limitation": "Только линия, 9 точек на направление; проверить методику и охват", "next_step": "ML Lead и Team Lead выбирают цель и временную проверку"},
    {"dataset": "context_observations.csv", "use": "Проверка масштаба и охвата", "limitation": "Иные объекты, без направления; не продолжение целевого ряда", "next_step": "Сверить определения с первоисточником"},
    {"dataset": "exports_clean.csv / exports_kaliningrad.csv", "use": "Описание внешней торговли; кандидат контекста", "limitation": "Два января, вложенные категории; нет подтверждённого OD и транспорта", "next_step": "Найти методику, публикацию, более длинную историю и дату доступности"},
    {"dataset": "transit_items_clean.csv / transit_monthly_summary.csv", "use": "Описание товарной выборки и качества полей", "limitation": "Группа 2204, конец 2016; нетто часто пусто; коридор и уникальность отправок не доказаны", "next_step": "Установить происхождение и критерии отбора; запросить нужный период и географию"},
    {"dataset": "scenario_parameters.csv", "use": "Справочные заявленные мощности объектов", "limitation": "Разные единицы, нет дат и доступности; не мощность всей цепочки", "next_step": "Аналитик подтверждает параметры выбранных действий"},
    {"dataset": "factors.csv", "use": "Пока только схема", "limitation": "Нет строк и подтверждённых доступных факторов", "next_step": "Заполнять после выбора цели и правил исторической доступности"},
    {"dataset": "Каталог, PDF, страницы, DOCX", "use": "Поиск источников и предметный контекст", "limitation": "Сохранённый материал не равен извлечённому ряду; расписание — план", "next_step": "Проверять первоисточники и извлекать совместимые показатели"},
])
display(suitability)
print("Готово к передаче: инвентаризация, источники, пропуски, проверки ключей, графики и ограничения.")


### Ваши заметки после просмотра

Дополните блок своими выводами. Человеческая приёмка отмечается только после реальной сверки с оригиналом.
ИИ подготовил этот ноутбук и автоматические проверки; это не подпись человека под методикой источника.


In [ ]:
review_notes = {
    "reviewer": "",                 # Кто просмотрел и сверил исходные ячейки.
    "review_date": "",              # Дата фактической проверки.
    "checked_sources_and_cells": "", # Какие файлы, листы, строки и единицы проверены.
    "findings": "",                 # Что вы увидели в таблицах и на графиках.
    "questions_for_team": "",       # Например: какой ряд выбрать для первой версии?
    "human_review_completed": False,
}
display(pd.DataFrame([review_notes]))


## 10. Сохранить обзор для передачи (по желанию)

По умолчанию эта ячейка ничего не записывает. Установите `SAVE_REVIEW = True`, чтобы сохранить
таблицы обзора, графики PNG и текстовый отчёт в `reports/generated/dataset_overview/<версия>/`.
Повторный запуск обновит только файлы этого обзора; исходники и подготовленные CSV не меняются.
Выводы ноутбука с реальными данными остаются в игнорируемой Git папке.


In [ ]:
SAVE_REVIEW = False
if SAVE_REVIEW:
    output_dir = ROOT / "reports/generated/dataset_overview" / manifest["data_version"]
    output_dir.mkdir(parents=True, exist_ok=True)
    review_tables = {"inventory": inventory, "missing_profile": missing_profile, "key_checks": key_checks,
                     "export_comparison": export_comparison, "transit_profile": transit_profile,
                     "suitability": suitability, "mass_checks": mass_checks}
    for name, frame in review_tables.items():
        frame.to_csv(output_dir / f"{name}.csv", index=False, encoding="utf-8-sig")
    for name, fig in figures.items():
        fig.savefig(output_dir / f"{name}.png", dpi=160, bbox_inches="tight")
    (output_dir / "review_notes.json").write_text(json.dumps(review_notes, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    report_lines = ["# Обзор датасетов", "", f"Версия данных: {manifest['data_version']}", "",
                    "Проверка комплекта: пройдена. Цель прогноза ожидает решения команды.", "",
                    "## Пригодность материалов", ""]
    for row in suitability.to_dict("records"):
        report_lines.extend([f"### {row['dataset']}", row["use"], row["limitation"], row["next_step"], ""])
    report_lines.extend(["## Человеческая проверка", json.dumps(review_notes, ensure_ascii=False, indent=2), "",
                         "ИИ подготовил ноутбук; статус проверки человеком задаётся после фактической сверки.", ""])
    (output_dir / "overview.md").write_text("\n".join(report_lines), encoding="utf-8")
    print("Обзор сохранён:", output_dir)
else:
    print("Сохранение выключено. Для экспорта таблиц, отчёта и графиков задайте SAVE_REVIEW = True.")


### Документы рядом

- `README.md` — ссылки на оригиналы и подготовленные файлы.
- `docs/contracts.md` — поля и единицы таблиц.
- `docs/tables-quality.md` — проверка четырёх пользовательских файлов.
- `docs/data-audit.md` — покрытие и варианты целевого ряда.
- `docs/data-preparation.md` — повторная сборка и передача оригиналов.

Для сохранения всего выполненного ноутбука и HTML без помещения реальных выводов в Git
используйте команды из `notebooks/README.md`.
